In [1]:
from pyspark.sql.functions import col, sum as _sum

customers = spark.table("bronze_customers")
orders = spark.table("bronze_orders")
items = spark.table("bronze_order_items")
payments = spark.table("bronze_payments")
products = spark.table("bronze_products")

# agregacja
payments_agg = (
    payments.groupBy("order_id")
    .agg(
        _sum("payment_value").alias("payment_value")
    )
)

# join
silver = (
    orders.alias("o")
    .join(customers.alias("c"), "customer_id", "left")
    .join(items.alias("i"), "order_id", "left")
    .join(products.alias("p"), "product_id", "left")
    .join(payments_agg.alias("pay"), "order_id", "left")
)

# finalne kolumny
silver_final = silver.select(
    col("order_id"),
    col("customer_id"),
    col("customer_city"),
    col("customer_state"),
    col("order_status"),
    col("order_purchase_timestamp"),
    col("product_id"),
    col("product_category_name"),
    col("price"),
    col("freight_value"),
    col("payment_value")
)

# output
(
    silver_final.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver_sales")
)

StatementMeta(, dec8ec4e-e5f5-4cc3-9723-04efc9ef7c71, 3, Finished, Available, Finished, False)